# VoiceVault · Whisper LoRA fine-tune for Hindi / Hinglish
Runs on a free **Colab T4** (Runtime → Change runtime type → T4 GPU). whisper-small: ~1-2 h for 3 epochs on ~10 h of audio.

Steps: install → data → baseline + train → export → before/after WER table.

In [ ]:
!git clone https://github.com/Aryan708/Voicevolt.git
%cd Voicevolt
!pip install -q -r finetune/requirements-finetune.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Data
Option A: public Hindi (FLEURS). Option B (recommended in addition): your own Hinglish clips in `my_clips/` with a `metadata.csv` (`file,text,speaker`).
Upload a zip of `my_clips/` to Colab and unzip it, or mount Google Drive.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# FLEURS has no speaker ids; its `id` column is the *sentence* id (read by several speakers),
# so grouping by it gives a sentence-disjoint split: no test sentence is seen in training.
!python finetune/prepare_data.py \
    --hf google/fleurs --hf-config hi_in --text-col transcription --speaker-col id \
    --trust-remote-code --out data/asr
# add your Hinglish recordings too:
# !python finetune/prepare_data.py --local my_clips --hf google/fleurs --hf-config hi_in \
#     --text-col transcription --trust-remote-code --out data/asr

## 2 · Train LoRA adapter

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/whisper-small-hinglish/checkpoints

In [ ]:
!python finetune/train_lora.py --manifest data/asr/manifest.jsonl \
    --base openai/whisper-small --language hindi --epochs 3 --batch 16 \
    --out runs/whisper-small-hinglish

## 3 · Merge + convert for faster-whisper

In [ ]:
!python finetune/export_model.py --base openai/whisper-small \
    --adapter runs/whisper-small-hinglish/adapter --out models/whisper-small-hinglish \
    --quantization int8_float16
# optional: --push YOUR_HF_USERNAME/whisper-small-hinglish   (run `huggingface-cli login` first)

## 4 · Before / after on the held-out test speakers

In [ ]:
!python finetune/evaluate_asr.py --manifest data/asr/manifest.jsonl --language hi \
    --model small --label whisper-small \
    --model models/whisper-small-hinglish/ct2 --label whisper-small-hinglish-lora \
    --device cuda --compute-type float16
from IPython.display import Markdown
Markdown(open("reports/asr/results.md").read())

## 5 · Use it in VoiceVault
Download `models/whisper-small-hinglish/ct2` and set `STT_MODEL=/path/to/ct2` in `.env` (live voice), or pass `--model /path/to/ct2` to `scripts/ingest.py` / `scripts/index.py` (recordings).

In [ ]:
!cd models && zip -qr whisper-small-hinglish-ct2.zip whisper-small-hinglish/ct2
from google.colab import files; files.download("models/whisper-small-hinglish-ct2.zip")